# v31 — Isu #3: Re-evaluasi Sebelum Re-entry Setelah TP ("Chasing")

**Konteks (dari brief perbaikan Isu Signal v13, isu #3)**: robot v13 belum re-evaluasi
kondisi market sebelum entry baru setelah posisi sebelumnya closed. Fokus yang dipilih user
(2026-09-01): kondisi **setelah TP (menang)** — begitu 1 posisi kena TP, guard "cuma 1 posisi
OPEN" (`list_open_tickets()` di `usecase.py`) langsung lepas, dan polling berikutnya bebas
ambil sinyal baru TANPA cek apakah entry baru itu "mengejar harga" (chasing) di arah yang sama
persis setelah baru saja profit besar — berisiko entry di titik yang sudah jauh dari area
wajar/momentum sudah mulai habis.

**Hipotesis yang diuji**: trade yang terjadi TEPAT SETELAH TP win, DI ARAH YANG SAMA, dan
entry price-nya sudah bergerak jauh (relatif ATR) dari entry price trade sebelumnya —
punya win rate/PF lebih buruk dibanding trade "segar" (bukan re-entry searah).

**Bukan mengulang riset regime (v20-v30)** — ini soal urutan/sekuens trade (trade N vs trade
N-1), bukan soal kondisi market independen per-trade.

**Base**: reuse persis engine `run_backtest_v28` (parameter LIVE AKTIF saat ini: v13 core +
S/R proximity filter, `sr_source="both"`, `sr_near_atr_mult=3.0`, `sr_strong_score_bonus=4.0`,
`sr_min_atr_for_breakout=2.1`) — supaya baseline yang dianalisis persis sama dengan yang jalan
live sekarang, cuma ditambah kolom `entry_price`/`exit_price`/`atr` di output trade utk
analisis sekuens (tidak mengubah logika trading engine v28 sama sekali).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v31"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v28 (sudah ada level S/R H1 & M15)

In [2]:
SR_CACHE_PATH = PROCESSED_DIR / "v28" / "df_2019_2026_sr.parquet"
assert SR_CACHE_PATH.exists(), "Cache v28 belum ada -- jalankan notebook v28 dulu"
df = pd.read_parquet(SR_CACHE_PATH)
print(df.shape)
df.head(2)

(518403, 24)


,datetime,open,high,low,close,adx,atr,v12_score,bull_chain,bear_chain,...,h1_ob_bull,h1_ob_bear,h1_ema_50,h1_ema_200,h1_bos_bull,h1_bos_bear,h1_sr_resistance,h1_sr_support,m15_sr_resistance,m15_sr_support
0,2019-01-01 23:00:00+00:00,1282.295,1282.345,1281.448,1281.448,0.000000,0.897000,1.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-01 23:05:00+00:00,1281.348,1281.548,1281.248,1281.548,7.142857,0.854357,-0.8,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Backtest engine v31 = v28 + kolom entry_price/exit_price/atr di output trade

Logika trading (kapan entry, kapan exit, kill-switch, filter OB/S/R/H1 alignment) **identik**
dengan `run_backtest_v28` — cuma menambah field ke `trades.append(...)` supaya bisa dianalisis
per-sekuens. Tidak ada perubahan behavior.

In [3]:
def check_h1_alignment_v28(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v31(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    sr_source: str = "both",
    sr_near_atr_mult: float = 3.0,
    sr_action: str = "skip",
    sr_strong_score_bonus: float = 4.0,
    sr_min_atr_for_breakout: float = 2.1,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = True,
    fixed_lot_value: float = 0.03,
    initial_equity_override: float = 2000.0,
    # --- Isu #3: filter anti-chasing setelah TP ---
    chase_filter_enabled: bool = False,
    chase_same_dir_atr_mult: float = 1.5,  # jarak entry baru vs entry sblmnya (dlm ATR) yg dianggap "sudah jauh/chasing"
    chase_strong_score_bonus: float = 4.0,  # skor sangat kuat -> filter chase diabaikan
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    peak_equity = base_equity
    last_trade = None  # {"direction", "entry_price", "result"} -- trade TERAKHIR yg closed, utk cek chasing
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v28(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points
        mode = "NORMAL"

        # --- Filter S/R proximity (identik v28) ---
        if sr_source != "none":
            opposing_level = None
            if direction == "BUY":
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_res_arr[i]):
                    candidates.append(h1_res_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_res_arr[i]):
                    candidates.append(m15_res_arr[i])
                if candidates:
                    opposing_level = min(candidates)
            else:
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_sup_arr[i]):
                    candidates.append(h1_sup_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_sup_arr[i]):
                    candidates.append(m15_sup_arr[i])
                if candidates:
                    opposing_level = max(candidates)

            if opposing_level is not None:
                dist_to_level = abs(opposing_level - close)
                is_near = dist_to_level <= (sr_near_atr_mult * atr)
                is_strong_signal = abs(score) >= (min_signal_score + sr_strong_score_bonus)
                is_breakout_atr = sr_min_atr_for_breakout is not None and atr >= sr_min_atr_for_breakout
                if is_near and not is_strong_signal and not is_breakout_atr:
                    if sr_action == "skip":
                        i += 1
                        continue
                    elif sr_action == "adjust":
                        buffer = 0.1 * atr
                        adjusted_tp = opposing_level - buffer if direction == "BUY" else opposing_level + buffer
                        if direction == "BUY" and adjusted_tp < tp_price and adjusted_tp > entry_price:
                            tp_price = adjusted_tp
                            mode = "SR_ADJUSTED"
                        elif direction == "SELL" and adjusted_tp > tp_price and adjusted_tp < entry_price:
                            tp_price = adjusted_tp
                            mode = "SR_ADJUSTED"

        # --- Filter Isu #3: anti-chasing setelah TP ---
        is_chase = False
        if chase_filter_enabled and last_trade is not None:
            same_dir = last_trade["direction"] == direction
            was_win = last_trade["result"] == "WIN"
            dist_from_last_entry = abs(entry_price - last_trade["entry_price"])
            moved_far = dist_from_last_entry >= (chase_same_dir_atr_mult * atr)
            is_strong_signal = abs(score) >= (min_signal_score + chase_strong_score_bonus)
            if same_dir and was_win and moved_far and not is_strong_signal:
                is_chase = True
        if is_chase:
            i += 1
            continue

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        peak_equity = max(peak_equity, equity)

        result = "WIN" if pnl > 0 else "LOSS"

        # sekuens: apakah trade INI sendiri adalah chase-candidate (dipakai analisis deskriptif
        # walau chase_filter_enabled=False, supaya bisa lihat perbandingan tanpa filter dulu)
        is_chase_candidate = False
        dist_from_last_entry_val = np.nan
        if last_trade is not None:
            same_dir = last_trade["direction"] == direction
            was_win = last_trade["result"] == "WIN"
            dist_from_last_entry_val = abs(entry_price - last_trade["entry_price"]) / atr
            if same_dir and was_win:
                is_chase_candidate = dist_from_last_entry_val >= chase_same_dir_atr_mult

        trades.append({
            "entry_time": entry_time, "exit_time": exit_time, "mode": mode, "direction": direction,
            "entry_price": entry_price, "exit_price": exit_price, "atr": atr, "score": score,
            "pnl": pnl, "result": result, "equity_after": equity,
            "is_chase_candidate": is_chase_candidate,
            "dist_from_last_entry_atr": dist_from_last_entry_val,
            "prev_result": last_trade["result"] if last_trade is not None else None,
            "prev_direction": last_trade["direction"] if last_trade is not None else None,
        })
        last_trade = {"direction": direction, "entry_price": entry_price, "result": result}
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v31 siap.")

Backtest engine v31 siap.


## 3. Baseline: jalankan dgn parameter LIVE AKTIF saat ini (v28), chase_filter OFF

Full period 2019-2026, fixed lot 0.03 & modal $2000 (konvensi full-period test session ini,
biar gak collapse ke equity negatif tanpa kill-switch).

In [4]:
trades_base = run_backtest_v31(df, chase_filter_enabled=False)
print(evaluate(trades_base, initial_equity=2000.0))
trades_base[["result", "is_chase_candidate"]].value_counts()

{'total_trades': 1754, 'win_rate_pct': 35.63, 'profit_factor': np.float64(1.25), 'net_pnl': np.float64(2967.73), 'max_drawdown_pct': np.float64(-116.84)}


result  is_chase_candidate
LOSS    False                 839
WIN     False                 388
LOSS    True                  290
WIN     True                  237
Name: count, dtype: int64

## 4. Uji hipotesis: apakah trade "chase candidate" (searah + setelah WIN + entry jauh dari
entry sblmnya) punya win rate/PF lebih buruk dibanding trade lain?

Catatan: kolom `is_chase_candidate` di sini dihitung dari trade `run_backtest_v31` dgn
`chase_filter_enabled=False` (jadi tidak ada trade yang di-skip krn filter chase) —
supaya perbandingan apple-to-apple: seandainya filter TIDAK ada, gimana hasil grup
chase-candidate vs bukan?

In [5]:
chase_grp = trades_base[trades_base["is_chase_candidate"]]
other_grp = trades_base[~trades_base["is_chase_candidate"]]

print(f"Chase-candidate: n={len(chase_grp)}, win_rate={round((chase_grp['pnl']>0).mean()*100,2)}%, "
      f"PF={round(chase_grp[chase_grp.pnl>0].pnl.sum()/abs(chase_grp[chase_grp.pnl<=0].pnl.sum()),2) if (chase_grp.pnl<=0).any() else float('inf')}, "
      f"avg_pnl={round(chase_grp['pnl'].mean(),3)}")
print(f"Lainnya:         n={len(other_grp)}, win_rate={round((other_grp['pnl']>0).mean()*100,2)}%, "
      f"PF={round(other_grp[other_grp.pnl>0].pnl.sum()/abs(other_grp[other_grp.pnl<=0].pnl.sum()),2) if (other_grp.pnl<=0).any() else float('inf')}, "
      f"avg_pnl={round(other_grp['pnl'].mean(),3)}")

u_stat, p_val = stats.mannwhitneyu(chase_grp["pnl"], other_grp["pnl"], alternative="two-sided")
print(f"\nMann-Whitney U pnl chase vs lainnya: U={u_stat:.1f}, p={p_val:.4f}")

chi2_table = pd.crosstab(trades_base["is_chase_candidate"], trades_base["result"])
print(chi2_table)
chi2, chi2_p, dof, expected = stats.chi2_contingency(chi2_table)
print(f"Chi-square win/loss vs chase-candidate: chi2={chi2:.2f}, p={chi2_p:.4f}")

Chase-candidate: n=527, win_rate=44.97%, PF=1.64, avg_pnl=5.268
Lainnya:         n=1227, win_rate=31.62%, PF=1.03, avg_pnl=0.156

Mann-Whitney U pnl chase vs lainnya: U=337822.0, p=0.1358
result              LOSS  WIN
is_chase_candidate           
False                839  388
True                 290  237
Chi-square win/loss vs chase-candidate: chi2=28.07, p=0.0000


## 5. Cek sensitivitas terhadap threshold jarak (chase_same_dir_atr_mult) & filter berdasar
SEMUA re-entry searah setelah WIN (bukan cuma yg "jauh") -- utk lihat apakah efeknya datang
dari jarak, atau cuma dari re-entry searah setelah WIN apapun jaraknya.

In [6]:
same_dir_after_win = trades_base[(trades_base["prev_result"] == "WIN") & (trades_base["prev_direction"] == trades_base["direction"])]
other_after_win_or_loss = trades_base[~((trades_base["prev_result"] == "WIN") & (trades_base["prev_direction"] == trades_base["direction"]))]

print(f"SEMUA re-entry searah setelah WIN (apapun jarak): n={len(same_dir_after_win)}, "
      f"win_rate={round((same_dir_after_win['pnl']>0).mean()*100,2)}%")
print(f"Lainnya:                                          n={len(other_after_win_or_loss)}, "
      f"win_rate={round((other_after_win_or_loss['pnl']>0).mean()*100,2)}%")

print("\n--- Breakdown win rate re-entry searah-setelah-WIN by jarak (kuartil ATR) ---")
sd = same_dir_after_win.dropna(subset=["dist_from_last_entry_atr"]).copy()
sd["dist_bucket"] = pd.qcut(sd["dist_from_last_entry_atr"], 4, duplicates="drop")
print(sd.groupby("dist_bucket", observed=True).agg(n=("pnl", "size"), win_rate=("pnl", lambda x: round((x>0).mean()*100, 2)), avg_pnl=("pnl", "mean")))

SEMUA re-entry searah setelah WIN (apapun jarak): n=584, win_rate=44.52%
Lainnya:                                          n=1170, win_rate=31.2%

--- Breakdown win rate re-entry searah-setelah-WIN by jarak (kuartil ATR) ---
                   n  win_rate    avg_pnl
dist_bucket                              
(0.0679, 2.723]  146     45.89   2.964158
(2.723, 4.769]   146     48.63   4.055278
(4.769, 8.248]   146     48.63  11.370942
(8.248, 42.031]  146     34.93  -0.387602


## 6. Kalau ada sinyal robust: grid search parameter filter (skip chase), train/test split

In [7]:
df["datetime"] = pd.to_datetime(df["datetime"])
TRAIN_END = pd.Timestamp("2024-01-01", tz=df["datetime"].dt.tz)
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"Train: {len(df_train)} candle ({df_train['datetime'].min()} - {df_train['datetime'].max()})")
print(f"Test:  {len(df_test)} candle ({df_test['datetime'].min()} - {df_test['datetime'].max()})")

Train: 351136 candle (2019-01-01 23:00:00+00:00 - 2023-12-29 21:55:00+00:00)
Test:  167267 candle (2024-01-01 23:00:00+00:00 - 2026-08-06 12:35:00+00:00)


In [8]:
baseline_train = run_backtest_v31(df_train, chase_filter_enabled=False)
baseline_test = run_backtest_v31(df_test, chase_filter_enabled=False)
print("Baseline TRAIN:", evaluate(baseline_train, initial_equity=2000.0))
print("Baseline TEST: ", evaluate(baseline_test, initial_equity=2000.0))

results = []
for atr_mult in [0.5, 1.0, 1.5, 2.0, 3.0]:
    for score_bonus in [2.0, 4.0, 999.0]:  # 999 = efektif "tidak pernah dianggap sangat kuat" (selalu difilter kalau chase)
        t_train = run_backtest_v31(df_train, chase_filter_enabled=True, chase_same_dir_atr_mult=atr_mult, chase_strong_score_bonus=score_bonus)
        m_train = evaluate(t_train, initial_equity=2000.0)
        results.append({"atr_mult": atr_mult, "score_bonus": score_bonus, **{f"train_{k}": v for k, v in m_train.items()}})

res_df = pd.DataFrame(results).sort_values("train_profit_factor", ascending=False)
res_df.head(15)

Baseline TRAIN: {'total_trades': 944, 'win_rate_pct': 23.2, 'profit_factor': np.float64(0.6), 'net_pnl': np.float64(-2024.44), 'max_drawdown_pct': np.float64(-101.22)}
Baseline TEST:  {'total_trades': 810, 'win_rate_pct': 50.12, 'profit_factor': np.float64(1.75), 'net_pnl': np.float64(4992.18), 'max_drawdown_pct': np.float64(-20.33)}


,atr_mult,score_bonus,train_total_trades,train_win_rate_pct,train_profit_factor,train_net_pnl,train_max_drawdown_pct
12,3.0,2.0,638,21.47,0.62,-1228.93,-61.45
9,2.0,2.0,600,20.83,0.59,-1248.11,-62.41
0,0.5,2.0,568,20.95,0.59,-1160.10,-58.01
14,3.0,999.0,451,18.40,0.59,-873.40,-43.67
13,3.0,4.0,470,18.51,0.58,-947.68,-47.38
6,1.5,2.0,587,20.95,0.58,-1228.45,-61.42
3,1.0,2.0,582,20.62,0.57,-1250.51,-62.53
11,2.0,999.0,422,17.30,0.54,-933.53,-46.68
7,1.5,4.0,410,17.32,0.53,-904.59,-45.23
10,2.0,4.0,441,17.46,0.52,-1008.71,-50.44


In [9]:
# Validasi kandidat terbaik train di TEST out-of-sample
top_candidates = res_df.head(5)
for _, row in top_candidates.iterrows():
    t_test = run_backtest_v31(df_test, chase_filter_enabled=True, chase_same_dir_atr_mult=row["atr_mult"], chase_strong_score_bonus=row["score_bonus"])
    m_test = evaluate(t_test, initial_equity=2000.0)
    print(f"atr_mult={row['atr_mult']}, score_bonus={row['score_bonus']} -> "
          f"TRAIN PF={row['train_profit_factor']}, n={row['train_total_trades']} | "
          f"TEST PF={m_test['profit_factor']}, n={m_test['total_trades']}, win_rate={m_test['win_rate_pct']}%")

atr_mult=3.0, score_bonus=2.0 -> TRAIN PF=0.62, n=638.0 | TEST PF=1.52, n=371, win_rate=48.52%


atr_mult=2.0, score_bonus=2.0 -> TRAIN PF=0.59, n=600.0 | TEST PF=1.59, n=327, win_rate=49.85%


atr_mult=0.5, score_bonus=2.0 -> TRAIN PF=0.59, n=568.0 | TEST PF=1.72, n=279, win_rate=49.46%


atr_mult=3.0, score_bonus=999.0 -> TRAIN PF=0.59, n=451.0 | TEST PF=1.86, n=203, win_rate=51.72%


atr_mult=3.0, score_bonus=4.0 -> TRAIN PF=0.58, n=470.0 | TEST PF=1.91, n=212, win_rate=51.89%


## 7. Klarifikasi ulang dari user (2026-09-01): bukan soal jarak antar-entry

User klarifikasi maksud Isu #3 sebenarnya: setelah TP, sebelum entry lagi searah, robot harus
cek ulang apakah harga SEKARANG sudah dekat S/R **DAN** ATR/Fibonacci menunjukkan momentum
cukup kuat utk breakout -- kalau tidak cukup kuat & dekat S/R, **skip** drpd kena SL (mantul).

**Temuan penting**: `check_sr_proximity()` yang SUDAH LIVE (v28) ternyata SUDAH otomatis
meng-cover kasus ini -- fungsi ini dipanggil di SETIAP `check_signal_and_trade()`, termasuk
setelah posisi sebelumnya baru TP (tidak ada logika "skip pengecekan S/R kalau re-entry").
Filter ini sudah pakai level S/R H1/M15 TERBARU (dihitung ulang tiap polling) + syarat ATR
breakout (>=2.1). Yang BELUM ada: **Fibonacci** sbg faktor tambahan.

**Uji di sini**: apakah menambahkan syarat Fibonacci (harga di zona retracement rawan
reversal, mis. dekat level 0.618/0.786 dari swing M5 50-candle) sbg filter TAMBAHAN di atas
S/R+ATR yang sudah ada, benar-benar memperbaiki filter yang sudah live -- bukan cuma utk kasus
setelah-TP, tapi general (krn filter S/R yang ada sudah general juga).

In [10]:
# Merge kolom Fibonacci M5 (dari data mentah v01, belum ada di cache v28) by datetime
fib_cols = ["datetime", "fib_swing_high", "fib_swing_low", "fib_236", "fib_382", "fib_500", "fib_618", "fib_786"]
df_fib_raw = pd.read_csv(
    PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME / "v01" / "xauusd_m5_full_indicators.csv",
    usecols=fib_cols,
)
df_fib_raw["datetime"] = pd.to_datetime(df_fib_raw["datetime"], utc=True)

df = df.merge(df_fib_raw, on="datetime", how="left")
print(df.shape)
print(df[fib_cols].isna().sum())
df[["close"] + fib_cols[1:]].tail(3)

(518403, 31)
datetime           0
fib_swing_high    49
fib_swing_low     49
fib_236           49
fib_382           49
fib_500           49
fib_618           49
fib_786           49
dtype: int64


,close,fib_swing_high,fib_swing_low,fib_236,fib_382,fib_500,fib_618,fib_786
518400,4261.595,4284.555,4253.035,4277.11628,4272.51436,4268.795,4265.07564,4259.78028
518401,4267.565,4284.555,4253.035,4277.11628,4272.51436,4268.795,4265.07564,4259.78028
518402,4269.235,4284.555,4253.035,4277.11628,4272.51436,4268.795,4265.07564,4259.78028


### Definisi "zona Fibonacci rawan reversal"

Swing high/low rolling 50-candle M5 (`fib_swing_high`/`fib_swing_low`) dipakai sbg acuan
range. Untuk sinyal **BUY** (harga naik, swing_low = titik awal gerakan naik): kalau harga
sudah retrace turun ke deket level dalam (0.618-0.786, dari swing_high) sebelum lanjut naik
lagi -- itu WAJAR (pullback sehat). Tapi kalau harga sudah **melewati/dekat swing_high itu
sendiri lagi** (retracement rendah, dekat 0.0-0.236 dari swing_high, artinya harga sudah balik
ke ujung atas range) -- itu candidate zona "sudah mentok", rawan mantul turun. Sebaliknya utk
SELL (dekat swing_low = sudah mentok bawah).

Jadi definisi "rawan" yang diuji: **jarak harga ke `fib_236` (retracement dangkal) dekat**, DAN
arah sinyal itu MENUJU ujung range yang sama (BUY menuju swing_high, SELL menuju swing_low)
-- gunakan ATR-relative distance spy konsisten dgn filter S/R yg sudah ada.

### Pendekatan lebih objektif: ulangi metodologi investigasi v28 (Mann-Whitney U)

Drpd menebak dulu bentuk filter Fibonacci, ulangi metodologi yang sudah terbukti di v28: ambil
SEMUA sinyal yang lolos threshold dasar & dekat S/R berlawanan (kandidat "rawan", SEBELUM
filter ATR-breakout diterapkan), beri label MANTUL (loss/reversal) vs TEMBUS (win/breakout),
lalu uji apakah jarak ke level Fibonacci terdekat (0.236/0.382/0.5/0.618/0.786) **signifikan
membedakan** kedua grup -- sama persis cara ATR ditemukan signifikan dulu. Kalau Fibonacci
TIDAK signifikan (spt jarak S/R & skor sinyal yang sudah terbukti tidak signifikan di v28),
maka tidak perlu ditambahkan sbg filter.

In [11]:
# Reuse cache investigasi v28: SEMUA sinyal near-S/R (2282 baris) dgn label MANTUL(LOSS)/TEMBUS(WIN)
near_sr = pd.read_parquet(PROCESSED_DIR / "v28" / "all_signals_with_sr_dist.parquet")
print(near_sr.shape)
print(near_sr["result"].value_counts())

# Join dgn close + level fib by entry_time, utk hitung jarak ke fib level terdekat (ATR-relative)
lookup = df[["datetime", "close"] + fib_cols[1:]].rename(columns={"datetime": "entry_time"})
near_sr = near_sr.merge(lookup, on="entry_time", how="left")
print("Missing fib after merge:", near_sr[fib_cols[1:]].isna().sum().sum())

for lvl in ["fib_236", "fib_382", "fib_500", "fib_618", "fib_786"]:
    near_sr[f"dist_{lvl}_atr"] = (near_sr["close"] - near_sr[lvl]).abs() / near_sr["atr"]

# jarak ke level fib TERDEKAT (dari 5 level), ATR-relative
near_sr["dist_nearest_fib_atr"] = near_sr[[f"dist_{lvl}_atr" for lvl in ["fib_236","fib_382","fib_500","fib_618","fib_786"]]].min(axis=1)
near_sr[["result", "dist_nearest_fib_atr"]].groupby("result").describe()

(2282, 9)
result
LOSS    1924
WIN      358
Name: count, dtype: int64
Missing fib after merge: 0


dist_nearest_fib_atr                                                                      
                      count      mean       std       min       25%       50%       75%       max
result                                                                                           
LOSS                 1924.0  1.059213  0.744710  0.000045  0.416828  0.979752  1.541411  4.701168
WIN                   358.0  0.931186  0.659015  0.005613  0.354242  0.871231  1.367489  3.235007

In [12]:
mantul = near_sr[near_sr["result"] == "LOSS"]
tembus = near_sr[near_sr["result"] == "WIN"]

u_stat, p_val = stats.mannwhitneyu(
    mantul["dist_nearest_fib_atr"].dropna(), tembus["dist_nearest_fib_atr"].dropna(), alternative="two-sided"
)
print(f"MANTUL (n={mantul['dist_nearest_fib_atr'].notna().sum()}): mean dist_nearest_fib_atr = {mantul['dist_nearest_fib_atr'].mean():.3f}")
print(f"TEMBUS (n={tembus['dist_nearest_fib_atr'].notna().sum()}): mean dist_nearest_fib_atr = {tembus['dist_nearest_fib_atr'].mean():.3f}")
print(f"Mann-Whitney U: U={u_stat:.1f}, p={p_val:.4f}")

# Cek juga per-level individual (siapa tau salah satu level spesifik signifikan meski gabungan tidak)
print("\n--- Per level Fibonacci individual ---")
for lvl in ["fib_236", "fib_382", "fib_500", "fib_618", "fib_786"]:
    col = f"dist_{lvl}_atr"
    u, p = stats.mannwhitneyu(mantul[col].dropna(), tembus[col].dropna(), alternative="two-sided")
    print(f"{lvl}: MANTUL mean={mantul[col].mean():.3f}, TEMBUS mean={tembus[col].mean():.3f}, p={p:.4f}")

MANTUL (n=1924): mean dist_nearest_fib_atr = 1.059
TEMBUS (n=358): mean dist_nearest_fib_atr = 0.931
Mann-Whitney U: U=375729.0, p=0.0062

--- Per level Fibonacci individual ---
fib_236: MANTUL mean=2.731, TEMBUS mean=2.882, p=0.2754
fib_382: MANTUL mean=2.852, TEMBUS mean=2.789, p=0.4847
fib_500: MANTUL mean=3.035, TEMBUS mean=2.798, p=0.0087
fib_618: MANTUL mean=3.294, TEMBUS mean=2.889, p=0.0002
fib_786: MANTUL mean=3.789, TEMBUS mean=3.172, p=0.0001


### Interpretasi hati-hati: arahnya TERBALIK dari hipotesis awal

`fib_618`, `fib_786`, `fib_500` signifikan (p<0.01) -- TAPI arahnya kebalikan dari dugaan
"dekat Fib = rawan mantul": trade MANTUL (loss) justru rata-rata LEBIH JAUH dari level fib
dalam (3.29-3.79x ATR) dibanding trade TEMBUS (win, 2.89-3.17x ATR). Effect size kecil (selisih
~0.1-0.6x ATR) meski p-value kecil -- kemungkinan besar krn n besar (1924 vs 358), bukan efek
praktis besar.

Jangan buru-buru simpulkan arah filter dari p-value/mean saja (pelajaran dari ATR: cuma
signifikansi + ARAH yang MASUK AKAL yang layak jadi filter). Uji langsung sbg filter ablation
di backtest -- baik arah "skip kalau JAUH dari fib dalam" maupun "skip kalau DEKAT fib dalam"
-- biar keputusan berbasis hasil out-of-sample, bukan asumsi arah.

## 8. Backtest engine v31c: v28 + syarat TAMBAHAN Fibonacci di filter S/R proximity

Filter S/R proximity yang sudah live: dekat S/R & TIDAK strong-score & TIDAK ATR-breakout ->
SKIP. Di sini ditambah 1 opsi baru `fib_gate` yang, KALAU aktif, menambah syarat lolos: hanya
boleh entry dekat-S/R (meski ATR breakout) kalau jarak ke level fib dalam terdekat (0.5/0.618/
0.786) juga di atas ambang tertentu (`fib_min_dist_atr`) -- sesuai arah yang ditemukan di atas
(MANTUL justru JAUH dari fib, jadi diuji arah "wajib CUKUP JAUH dari fib utk lolos", bukan
"wajib dekat").

In [13]:
h1_res_arr_full = df["h1_sr_resistance"].to_numpy()
h1_sup_arr_full = df["h1_sr_support"].to_numpy()
m15_res_arr_full = df["m15_sr_resistance"].to_numpy()
m15_sup_arr_full = df["m15_sr_support"].to_numpy()


def run_backtest_v31c(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    sr_source: str = "both",
    sr_near_atr_mult: float = 3.0,
    sr_strong_score_bonus: float = 4.0,
    sr_min_atr_for_breakout: float = 2.1,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = True,
    fixed_lot_value: float = 0.03,
    initial_equity_override: float = 2000.0,
    # --- filter Fibonacci tambahan (hanya berlaku KETIKA sudah dekat S/R & lolos via ATR-breakout) ---
    fib_gate_enabled: bool = False,
    fib_min_dist_atr: float = 1.0,   # syarat MINIMAL jarak ke fib dalam terdekat (ATR-relative) utk tetap lolos
    fib_direction: str = "far",      # "far"=wajib >=fib_min_dist_atr, "near"=wajib <=fib_min_dist_atr
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    fib_500_arr = df_signals["fib_500"].to_numpy()
    fib_618_arr = df_signals["fib_618"].to_numpy()
    fib_786_arr = df_signals["fib_786"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    peak_equity = base_equity
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v28(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        # --- Filter S/R proximity (identik v28, sr_action="skip" saja) ---
        if sr_source != "none":
            opposing_level = None
            if direction == "BUY":
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_res_arr[i]):
                    candidates.append(h1_res_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_res_arr[i]):
                    candidates.append(m15_res_arr[i])
                if candidates:
                    opposing_level = min(candidates)
            else:
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_sup_arr[i]):
                    candidates.append(h1_sup_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_sup_arr[i]):
                    candidates.append(m15_sup_arr[i])
                if candidates:
                    opposing_level = max(candidates)

            if opposing_level is not None:
                dist_to_level = abs(opposing_level - close)
                is_near = dist_to_level <= (sr_near_atr_mult * atr)
                is_strong_signal = abs(score) >= (min_signal_score + sr_strong_score_bonus)
                is_breakout_atr = sr_min_atr_for_breakout is not None and atr >= sr_min_atr_for_breakout
                if is_near and not is_strong_signal and not is_breakout_atr:
                    i += 1
                    continue
                # --- Filter Fibonacci TAMBAHAN: hanya dievaluasi utk kasus yg near-S/R & lolos krn ATR/skor ---
                if fib_gate_enabled and is_near:
                    fib_dists = []
                    for fib_arr in (fib_500_arr, fib_618_arr, fib_786_arr):
                        if np.isfinite(fib_arr[i]):
                            fib_dists.append(abs(close - fib_arr[i]) / atr)
                    if fib_dists:
                        nearest_fib_dist = min(fib_dists)
                        if fib_direction == "far" and nearest_fib_dist < fib_min_dist_atr:
                            i += 1
                            continue
                        if fib_direction == "near" and nearest_fib_dist > fib_min_dist_atr:
                            i += 1
                            continue

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        peak_equity = max(peak_equity, equity)

        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)

print("Engine v31c siap.")

Engine v31c siap.


In [14]:
# PENTING: df_train/df_test lama (section 6) dibuat SEBELUM merge kolom fib -- regenerate
# supaya df_train/df_test skrg punya kolom fib_500/fib_618/fib_786 yg dibutuhkan v31c.
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print("fib_500" in df_train.columns, "fib_500" in df_test.columns)

True True


In [15]:
baseline_train_c = run_backtest_v31c(df_train, fib_gate_enabled=False)
baseline_test_c = run_backtest_v31c(df_test, fib_gate_enabled=False)
print("Baseline (=v28 live) TRAIN:", evaluate(baseline_train_c, initial_equity=2000.0))
print("Baseline (=v28 live) TEST: ", evaluate(baseline_test_c, initial_equity=2000.0))

fib_results = []
for direction in ["far", "near"]:
    for min_dist in [0.5, 1.0, 1.5, 2.0, 3.0]:
        t_train = run_backtest_v31c(df_train, fib_gate_enabled=True, fib_min_dist_atr=min_dist, fib_direction=direction)
        m_train = evaluate(t_train, initial_equity=2000.0)
        fib_results.append({"fib_direction": direction, "fib_min_dist_atr": min_dist, **{f"train_{k}": v for k, v in m_train.items()}})

fib_res_df = pd.DataFrame(fib_results).sort_values("train_profit_factor", ascending=False)
fib_res_df

Baseline (=v28 live) TRAIN: {'total_trades': 944, 'win_rate_pct': 23.2, 'profit_factor': np.float64(0.6), 'net_pnl': np.float64(-2024.44), 'max_drawdown_pct': np.float64(-101.22)}
Baseline (=v28 live) TEST:  {'total_trades': 810, 'win_rate_pct': 50.12, 'profit_factor': np.float64(1.75), 'net_pnl': np.float64(4992.18), 'max_drawdown_pct': np.float64(-20.33)}


,fib_direction,fib_min_dist_atr,train_total_trades,train_win_rate_pct,train_profit_factor,train_net_pnl,train_max_drawdown_pct
9,near,3.0,928,23.17,0.61,-1938.95,-96.95
8,near,2.0,895,22.68,0.61,-1808.64,-90.43
6,near,1.0,837,21.62,0.60,-1676.08,-83.80
7,near,1.5,865,22.08,0.60,-1776.44,-88.82
5,near,0.5,820,20.98,0.58,-1745.42,-87.27
0,far,0.5,912,22.59,0.58,-2023.55,-101.18
1,far,1.0,892,21.86,0.56,-2087.55,-104.38
2,far,1.5,862,21.35,0.56,-1973.06,-98.65
3,far,2.0,841,20.81,0.55,-1949.60,-97.48
4,far,3.0,803,19.93,0.54,-1883.10,-94.16


In [16]:
# Validasi kandidat terbaik (train) di TEST out-of-sample, bandingkan ke baseline (=v28 live, tanpa fib gate)
print(f"Baseline TEST (tanpa fib gate): PF={evaluate(baseline_test_c, initial_equity=2000.0)['profit_factor']}, "
      f"n={evaluate(baseline_test_c, initial_equity=2000.0)['total_trades']}, "
      f"win_rate={evaluate(baseline_test_c, initial_equity=2000.0)['win_rate_pct']}%\n")

top_fib_candidates = fib_res_df.head(5)
for _, row in top_fib_candidates.iterrows():
    t_test = run_backtest_v31c(df_test, fib_gate_enabled=True, fib_min_dist_atr=row["fib_min_dist_atr"], fib_direction=row["fib_direction"])
    m_test = evaluate(t_test, initial_equity=2000.0)
    print(f"direction={row['fib_direction']}, min_dist={row['fib_min_dist_atr']} -> "
          f"TRAIN PF={row['train_profit_factor']}, n={row['train_total_trades']} | "
          f"TEST PF={m_test['profit_factor']}, n={m_test['total_trades']}, win_rate={m_test['win_rate_pct']}%")

Baseline TEST (tanpa fib gate): PF=1.75, n=810, win_rate=50.12%



direction=near, min_dist=3.0 -> TRAIN PF=0.61, n=928 | TEST PF=1.71, n=729, win_rate=48.97%


direction=near, min_dist=2.0 -> TRAIN PF=0.61, n=895 | TEST PF=1.68, n=646, win_rate=47.99%


direction=near, min_dist=1.0 -> TRAIN PF=0.6, n=837 | TEST PF=1.8, n=546, win_rate=47.44%


direction=near, min_dist=1.5 -> TRAIN PF=0.6, n=865 | TEST PF=1.71, n=608, win_rate=47.7%


direction=near, min_dist=0.5 -> TRAIN PF=0.58, n=820 | TEST PF=1.73, n=488, win_rate=46.72%


## 9. Isu #2 (percobaan baru): adx_min M5 DINAMIS berdasarkan kondisi ADX H1

**Klarifikasi user (2026-09-01)**: bukan soal `adx_min` tetap dinaikkan/diturunkan secara
global, dan bukan soal v29 (regime H1/M15 sbg pengali S/R filter, sudah dicoba & gagal
signifikan). Ide baru: **kalau H1 sideways (ADX H1 rendah), keputusan entry M5 jadi
"terlambat"** -- sinyal M5 yang lolos `adx_min=18` cuma osilasi kecil di dalam range besar H1
yang sideways, jadi meski kadang sempat profit sebentar, akhirnya kalah/whipsaw balik. Diamati
langsung dari data live 1 Sept 2026 (4 loss beruntun semua di ADX M5 ~21).

**Hipotesis diuji**: `adx_min` M5 harus lebih TINGGI (lebih ketat) saat ADX H1 rendah
(sideways), dan tetap normal saat ADX H1 tinggi (trending) -- konsep sama dgn v29 (regime H1
sbg modifier) TAPI diterapkan ke `adx_min` (gerbang paling awal, sebelum filter S/R), bukan ke
`sr_near_atr_mult` dst (gerbang lapis kedua yg sudah dicoba v29).

**Base**: pakai cache v29 (`df_2019_2026_regime.parquet`) yg sudah py `h1_regime_adx` (ADX H1,
no-lookahead via merge_asof) + level S/R H1/M15 dari v28."

In [17]:
REGIME_CACHE_PATH = PROCESSED_DIR / "v29" / "df_2019_2026_regime.parquet"
assert REGIME_CACHE_PATH.exists(), "Cache v29 belum ada"
df_regime = pd.read_parquet(REGIME_CACHE_PATH)
df_regime["datetime"] = pd.to_datetime(df_regime["datetime"])
print(df_regime.shape)
print(df_regime[["datetime", "adx", "h1_regime_adx", "m15_regime_adx"]].describe())

df_regime_train = df_regime[df_regime["datetime"] < TRAIN_END].reset_index(drop=True)
df_regime_test = df_regime[df_regime["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"Train: {len(df_regime_train)}, Test: {len(df_regime_test)}")

(518403, 28)


                 adx  h1_regime_adx  m15_regime_adx
count  518403.000000  518391.000000   518400.000000
mean       24.826185      26.956729       25.664775
std        10.501662      11.235206       10.787891
min         0.000000       0.000000        0.000000
25%        17.027933      18.513079       17.523747
50%        22.572038      24.736419       23.527499
75%        30.449570      33.356363       31.743005
max        83.850744      76.657500       77.588602


Train: 351136, Test: 167267


In [18]:
def run_backtest_v31d(
    df_signals: pd.DataFrame,
    adx_min_base: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    sr_source: str = "both",
    sr_near_atr_mult: float = 3.0,
    sr_strong_score_bonus: float = 4.0,
    sr_min_atr_for_breakout: float = 2.1,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = True,
    fixed_lot_value: float = 0.03,
    initial_equity_override: float = 2000.0,
    # --- adx_min M5 dinamis berdasar ADX H1 ---
    dynamic_adx_enabled: bool = False,
    h1_adx_sideways_threshold: float = 20.0,  # H1 ADX di bawah ini dianggap sideways
    adx_min_when_sideways: float = 25.0,      # adx_min M5 dipakai KETIKA H1 sideways (lebih ketat dari base)
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    h1_regime_adx_arr = df_signals["h1_regime_adx"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    peak_equity = base_equity
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue

        adx_min = adx_min_base
        if dynamic_adx_enabled:
            h1_adx = h1_regime_adx_arr[i]
            if np.isfinite(h1_adx) and h1_adx < h1_adx_sideways_threshold:
                adx_min = adx_min_when_sideways

        if adx < adx_min:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v28(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        if sr_source != "none":
            opposing_level = None
            if direction == "BUY":
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_res_arr[i]):
                    candidates.append(h1_res_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_res_arr[i]):
                    candidates.append(m15_res_arr[i])
                if candidates:
                    opposing_level = min(candidates)
            else:
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_sup_arr[i]):
                    candidates.append(h1_sup_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_sup_arr[i]):
                    candidates.append(m15_sup_arr[i])
                if candidates:
                    opposing_level = max(candidates)

            if opposing_level is not None:
                dist_to_level = abs(opposing_level - close)
                is_near = dist_to_level <= (sr_near_atr_mult * atr)
                is_strong_signal = abs(score) >= (min_signal_score + sr_strong_score_bonus)
                is_breakout_atr = sr_min_atr_for_breakout is not None and atr >= sr_min_atr_for_breakout
                if is_near and not is_strong_signal and not is_breakout_atr:
                    i += 1
                    continue

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        peak_equity = max(peak_equity, equity)

        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)

print("Engine v31d siap.")

Engine v31d siap.


In [19]:
baseline_train_d = run_backtest_v31d(df_regime_train, dynamic_adx_enabled=False)
baseline_test_d = run_backtest_v31d(df_regime_test, dynamic_adx_enabled=False)
print("Baseline (=v28 live, adx_min=18 tetap) TRAIN:", evaluate(baseline_train_d, initial_equity=2000.0))
print("Baseline (=v28 live, adx_min=18 tetap) TEST: ", evaluate(baseline_test_d, initial_equity=2000.0))

dyn_results = []
for h1_thresh in [15.0, 18.0, 20.0, 22.0, 25.0]:
    for adx_min_sw in [20.0, 22.0, 25.0, 28.0, 30.0]:
        t_train = run_backtest_v31d(
            df_regime_train, dynamic_adx_enabled=True,
            h1_adx_sideways_threshold=h1_thresh, adx_min_when_sideways=adx_min_sw,
        )
        m_train = evaluate(t_train, initial_equity=2000.0)
        dyn_results.append({
            "h1_adx_sideways_threshold": h1_thresh, "adx_min_when_sideways": adx_min_sw,
            **{f"train_{k}": v for k, v in m_train.items()},
        })

dyn_res_df = pd.DataFrame(dyn_results).sort_values("train_profit_factor", ascending=False)
dyn_res_df.head(15)

Baseline (=v28 live, adx_min=18 tetap) TRAIN: {'total_trades': 944, 'win_rate_pct': 23.2, 'profit_factor': np.float64(0.6), 'net_pnl': np.float64(-2024.44), 'max_drawdown_pct': np.float64(-101.22)}
Baseline (=v28 live, adx_min=18 tetap) TEST:  {'total_trades': 810, 'win_rate_pct': 50.12, 'profit_factor': np.float64(1.75), 'net_pnl': np.float64(4992.18), 'max_drawdown_pct': np.float64(-20.33)}


,h1_adx_sideways_threshold,adx_min_when_sideways,train_total_trades,train_win_rate_pct,train_profit_factor,train_net_pnl,train_max_drawdown_pct
6,18.0,22.0,924,23.59,0.62,-1896.82,-94.84
0,15.0,20.0,942,23.25,0.61,-1985.81,-99.29
1,15.0,22.0,938,23.45,0.61,-1962.38,-98.12
5,18.0,20.0,937,23.27,0.61,-1966.48,-98.32
4,15.0,30.0,911,23.27,0.61,-1939.21,-96.96
16,22.0,22.0,910,23.74,0.61,-1874.80,-93.74
15,22.0,20.0,937,23.37,0.61,-1952.41,-97.62
11,20.0,22.0,918,23.53,0.61,-1929.56,-96.48
10,20.0,20.0,937,23.27,0.61,-1959.45,-97.97
21,25.0,22.0,903,23.59,0.61,-1870.24,-93.51


In [20]:
# Validasi kandidat terbaik (train) di TEST out-of-sample
print(f"Baseline TEST: PF={evaluate(baseline_test_d, initial_equity=2000.0)['profit_factor']}, "
      f"n={evaluate(baseline_test_d, initial_equity=2000.0)['total_trades']}, "
      f"win_rate={evaluate(baseline_test_d, initial_equity=2000.0)['win_rate_pct']}%\n")

top_dyn_candidates = dyn_res_df.head(8)
for _, row in top_dyn_candidates.iterrows():
    t_test = run_backtest_v31d(
        df_regime_test, dynamic_adx_enabled=True,
        h1_adx_sideways_threshold=row["h1_adx_sideways_threshold"], adx_min_when_sideways=row["adx_min_when_sideways"],
    )
    m_test = evaluate(t_test, initial_equity=2000.0)
    print(f"h1_thresh={row['h1_adx_sideways_threshold']}, adx_min_sw={row['adx_min_when_sideways']} -> "
          f"TRAIN PF={row['train_profit_factor']}, n={row['train_total_trades']} | "
          f"TEST PF={m_test['profit_factor']}, n={m_test['total_trades']}, win_rate={m_test['win_rate_pct']}%")

Baseline TEST: PF=1.75, n=810, win_rate=50.12%



h1_thresh=18.0, adx_min_sw=22.0 -> TRAIN PF=0.62, n=924.0 | TEST PF=1.69, n=780, win_rate=49.1%


h1_thresh=15.0, adx_min_sw=20.0 -> TRAIN PF=0.61, n=942.0 | TEST PF=1.73, n=805, win_rate=49.69%


h1_thresh=15.0, adx_min_sw=22.0 -> TRAIN PF=0.61, n=938.0 | TEST PF=1.73, n=797, win_rate=49.44%


h1_thresh=18.0, adx_min_sw=20.0 -> TRAIN PF=0.61, n=937.0 | TEST PF=1.73, n=802, win_rate=49.63%


h1_thresh=15.0, adx_min_sw=30.0 -> TRAIN PF=0.61, n=911.0 | TEST PF=1.76, n=765, win_rate=49.93%


h1_thresh=22.0, adx_min_sw=22.0 -> TRAIN PF=0.61, n=910.0 | TEST PF=1.72, n=756, win_rate=49.6%


h1_thresh=22.0, adx_min_sw=20.0 -> TRAIN PF=0.61, n=937.0 | TEST PF=1.72, n=799, win_rate=49.56%


h1_thresh=20.0, adx_min_sw=22.0 -> TRAIN PF=0.61, n=918.0 | TEST PF=1.7, n=772, win_rate=49.22%


## 10. Investigasi live: kenapa loss beruntun 1 Sept 2026 terjadi?

Cross-check trade log live HFM (`trade_log_v13.csv`) thd data H1/M5 real via MT5 utk 8 trade
SELL 31 Agu - 1 Sept 2026 (5 loss beruntun jam 02:15-11:10 UTC). Hasil:
- **BUKAN H1 sideways** -- H1 ADX 31-43 (trending KUAT), H1 trend DOWN konsisten di semua trade.
- **BUKAN S/R proximity** -- jarak ke H1 support 3.18x-11.12x ATR (di luar radius filter aktif
  `sr_near_atr_mult=3.0`), filter v28 TIDAK triggered utk kasus2 ini.
- **POLA YANG KONSISTEN: `bear_chain` (momentum exhaustion M5) sudah 7-8/8** di 5 dari 6 loss
  beruntun -- robot entry SELL berulang kali (jarak antar entry cuma 1-2 jam) padahal tren M5
  sudah "matang"/exhaustion penuh. 2 trade WIN yang ada justru di bear_chain lebih rendah (6).
  3 dari 5 loss beruntun bahkan `is_exhausted=True` di CSV (SL/TP SUDAH diperkecil sesuai
  mekanisme v13 yg ada) -- TAPI tetap kena SL, artinya SL yang diperkecil pun tidak cukup
  melindungi dari retrace tajam saat tren sedang exhaustion.

**Cek riwayat**: v13 asli (`v13_momentum_exhaustion.ipynb`) SUDAH menguji SKIP vs REVERSE vs
SMALL-PROFIT saat chain 8/8, tapi di data TEST lama (2026-03 s/d 2026-08, sebelum HFM live):
SKIP PF=3.85 (TERTINGGI) vs SMALL-PROFIT PF=3.57, tapi SMALL-PROFIT menang di net_pnl (1179 vs
1084, krn SKIP kehilangan 51 trade/145 vs 196). Keputusan waktu itu pilih SMALL-PROFIT demi
profit lebih tinggi, TRADE-OFF profit factor sedikit lebih rendah -- bukan krn SKIP terbukti
buruk.

**Uji ulang di sini**: SKIP vs SMALL-PROFIT (kondisi live skrg) di data FULL 2019-2026 + filter
S/R v28 sudah aktif, utk lihat apakah kesimpulan itu masih berlaku dgn data lebih lengkap &
kondisi filter yang sudah berubah sejak itu.

In [21]:
def run_backtest_v31e(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    sr_source: str = "both",
    sr_near_atr_mult: float = 3.0,
    sr_strong_score_bonus: float = 4.0,
    sr_min_atr_for_breakout: float = 2.1,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = True,
    fixed_lot_value: float = 0.03,
    initial_equity_override: float = 2000.0,
    # --- momentum chain exhaustion ---
    exhaustion_chain_threshold: float = 8.0,
    exhaustion_mode: str = "small_profit",  # "small_profit" (spt live skrg) atau "skip" (uji ulang)
    exhaustion_sl_mult: float = 1.25,
    exhaustion_tp_mult: float = 1.0,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    bull_chain_arr = df_signals["bull_chain"].to_numpy()
    bear_chain_arr = df_signals["bear_chain"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    peak_equity = base_equity
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v28(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        if sr_source != "none":
            opposing_level = None
            if direction == "BUY":
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_res_arr[i]):
                    candidates.append(h1_res_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_res_arr[i]):
                    candidates.append(m15_res_arr[i])
                if candidates:
                    opposing_level = min(candidates)
            else:
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_sup_arr[i]):
                    candidates.append(h1_sup_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_sup_arr[i]):
                    candidates.append(m15_sup_arr[i])
                if candidates:
                    opposing_level = max(candidates)

            if opposing_level is not None:
                dist_to_level = abs(opposing_level - close)
                is_near = dist_to_level <= (sr_near_atr_mult * atr)
                is_strong_signal = abs(score) >= (min_signal_score + sr_strong_score_bonus)
                is_breakout_atr = sr_min_atr_for_breakout is not None and atr >= sr_min_atr_for_breakout
                if is_near and not is_strong_signal and not is_breakout_atr:
                    i += 1
                    continue

        dominant_chain = bull_chain_arr[i] if direction == "BUY" else bear_chain_arr[i]
        is_exhausted = np.isfinite(dominant_chain) and dominant_chain >= exhaustion_chain_threshold

        if is_exhausted and exhaustion_mode == "skip":
            i += 1
            continue

        if is_exhausted and exhaustion_mode == "small_profit":
            cur_sl_mult, cur_tp_mult = exhaustion_sl_mult, exhaustion_tp_mult
        else:
            cur_sl_mult, cur_tp_mult = sl_mult, tp_mult

        sl_points = cur_sl_mult * atr
        tp_points = cur_tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        peak_equity = max(peak_equity, equity)

        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
            "is_exhausted": is_exhausted,
        })
        i = next_i

    return pd.DataFrame(trades)

print("Engine v31e siap.")

Engine v31e siap.


In [22]:
small_profit_train = run_backtest_v31e(df_train, exhaustion_mode="small_profit")
skip_train = run_backtest_v31e(df_train, exhaustion_mode="skip")
small_profit_test = run_backtest_v31e(df_test, exhaustion_mode="small_profit")
skip_test = run_backtest_v31e(df_test, exhaustion_mode="skip")

print("=== TRAIN ===")
print("SMALL-PROFIT (spt live skrg):", evaluate(small_profit_train, initial_equity=2000.0))
print("SKIP saat exhaustion:        ", evaluate(skip_train, initial_equity=2000.0))
print()
print("=== TEST ===")
print("SMALL-PROFIT (spt live skrg):", evaluate(small_profit_test, initial_equity=2000.0))
print("SKIP saat exhaustion:        ", evaluate(skip_test, initial_equity=2000.0))

exh_trades = small_profit_test[small_profit_test["is_exhausted"]]
non_exh_trades = small_profit_test[~small_profit_test["is_exhausted"]]
print()
print("--- TEST small_profit breakdown ---")
exh_win = round((exh_trades["pnl"] > 0).mean() * 100, 2)
exh_avg = round(exh_trades["pnl"].mean(), 3)
print(f"Exhausted (n={len(exh_trades)}): win_rate={exh_win}%, avg_pnl={exh_avg}")
non_exh_win = round((non_exh_trades["pnl"] > 0).mean() * 100, 2)
non_exh_avg = round(non_exh_trades["pnl"].mean(), 3)
print(f"Non-exhausted (n={len(non_exh_trades)}): win_rate={non_exh_win}%, avg_pnl={non_exh_avg}")

=== TRAIN ===
SMALL-PROFIT (spt live skrg): {'total_trades': 971, 'win_rate_pct': 22.76, 'profit_factor': np.float64(0.58), 'net_pnl': np.float64(-2099.21), 'max_drawdown_pct': np.float64(-104.96)}
SKIP saat exhaustion:         {'total_trades': 767, 'win_rate_pct': 24.25, 'profit_factor': np.float64(0.64), 'net_pnl': np.float64(-1468.61), 'max_drawdown_pct': np.float64(-73.43)}

=== TEST ===
SMALL-PROFIT (spt live skrg): {'total_trades': 830, 'win_rate_pct': 50.24, 'profit_factor': np.float64(1.68), 'net_pnl': np.float64(4294.87), 'max_drawdown_pct': np.float64(-18.91)}
SKIP saat exhaustion:         {'total_trades': 633, 'win_rate_pct': 51.66, 'profit_factor': np.float64(1.88), 'net_pnl': np.float64(4462.92), 'max_drawdown_pct': np.float64(-15.27)}

--- TEST small_profit breakdown ---
Exhausted (n=208): win_rate=47.12%, avg_pnl=0.186
Non-exhausted (n=622): win_rate=51.29%, avg_pnl=6.843


## 11. Validasi statistik: Monte Carlo drawdown + Deflated Sharpe Ratio (SKIP vs SMALL-PROFIT)

Sesuai standing constraint proyek: sebelum diterapkan ke live, validasi risiko urutan trade (Monte Carlo reshuffle) dan signifikansi statistik (DSR) -- pola yang sama dipakai utk v28.

In [23]:
def monte_carlo_drawdown(pnls, initial_equity, n_sims=10000, seed=42):
    rng = np.random.default_rng(seed)
    max_dds = np.empty(n_sims)
    for sim in range(n_sims):
        shuffled = rng.permutation(pnls)
        equity_curve = initial_equity + np.cumsum(shuffled)
        running_max = np.maximum.accumulate(np.concatenate([[initial_equity], equity_curve]))
        dd = (np.concatenate([[initial_equity], equity_curve]) - running_max) / running_max * 100
        max_dds[sim] = dd.min()
    return max_dds

pnls_small = small_profit_test['pnl'].to_numpy()
pnls_skip = skip_test['pnl'].to_numpy()

mc_small = monte_carlo_drawdown(pnls_small, 2000.0)
mc_skip = monte_carlo_drawdown(pnls_skip, 2000.0)

print('=== Monte Carlo max drawdown (10.000 simulasi reshuffle), TEST period ===')
for label, mc, actual, n in [
    ('SMALL-PROFIT (live skrg)', mc_small, evaluate(small_profit_test, 2000.0)['max_drawdown_pct'], len(pnls_small)),
    ('SKIP', mc_skip, evaluate(skip_test, 2000.0)['max_drawdown_pct'], len(pnls_skip)),
]:
    print(f'--- {label} (n={n} trade) ---')
    print(f'  Actual historical max DD: {actual:.2f}%')
    print(f'  MC median: {np.median(mc):.2f}%')
    print(f'  MC P95 (5th pct): {np.percentile(mc, 5):.2f}%')
    print(f'  MC P99 (1st pct): {np.percentile(mc, 1):.2f}%')
    print(f'  MC worst: {mc.min():.2f}%')

print('Kesimpulan MC:')
if np.percentile(mc_skip, 5) > np.percentile(mc_small, 5):
    print('  SKIP secara risiko urutan LEBIH AMAN (P95 drawdown lebih dangkal) drpd SMALL-PROFIT.')
else:
    print('  SMALL-PROFIT secara risiko urutan lebih aman (P95 drawdown lebih dangkal) drpd SKIP.')

=== Monte Carlo max drawdown (10.000 simulasi reshuffle), TEST period ===
--- SMALL-PROFIT (live skrg) (n=830 trade) ---
  Actual historical max DD: -18.91%
  MC median: -8.89%
  MC P95 (5th pct): -15.62%
  MC P99 (1st pct): -20.25%
  MC worst: -30.81%
--- SKIP (n=633 trade) ---
  Actual historical max DD: -15.27%
  MC median: -8.05%
  MC P95 (5th pct): -14.25%
  MC P99 (1st pct): -18.53%
  MC worst: -28.15%
Kesimpulan MC:
  SKIP secara risiko urutan LEBIH AMAN (P95 drawdown lebih dangkal) drpd SMALL-PROFIT.


In [24]:
from scipy.stats import norm, skew, kurtosis

def daily_returns_from_trades(trades, initial_equity):
    trades = trades.copy()
    trades['entry_time'] = pd.to_datetime(trades['entry_time'])
    trades['date'] = trades['entry_time'].dt.date
    daily_pnl = trades.groupby('date')['pnl'].sum()
    return daily_pnl / initial_equity

def sharpe_ratio(returns):
    if returns.std() == 0:
        return 0.0
    return returns.mean() / returns.std() * np.sqrt(252)

def expected_max_sharpe(n_trials, sr_std=1.0):
    euler_gamma = 0.5772156649
    if n_trials <= 1:
        return 0.0
    return sr_std * ((1 - euler_gamma) * norm.ppf(1 - 1/n_trials) + euler_gamma * norm.ppf(1 - 1/(n_trials * np.e)))

def deflated_sharpe_ratio(sr, sr_benchmark, n_obs, skew_val, kurt_val):
    if n_obs <= 1:
        return 0.0
    denom = np.sqrt(1 - skew_val * sr + (kurt_val - 1) / 4 * sr**2)
    if denom <= 0:
        return 0.0
    return norm.cdf((sr - sr_benchmark) * np.sqrt(n_obs - 1) / denom)

returns_small = daily_returns_from_trades(small_profit_test, 2000.0)
returns_skip = daily_returns_from_trades(skip_test, 2000.0)

sr_small = sharpe_ratio(returns_small)
sr_skip = sharpe_ratio(returns_skip)
print(f'Sharpe Ratio (TEST period, annualized): SMALL-PROFIT={sr_small:.4f}, SKIP={sr_skip:.4f}')

# N_TRIALS kecil krn ini cuma 1 keputusan biner (mode exhaustion), bukan grid search besar spt v28
N_TRIALS = 2
sr_benchmark = expected_max_sharpe(N_TRIALS, sr_std=returns_skip.std() * np.sqrt(252) if returns_skip.std() > 0 else 1.0)
skew_skip = skew(returns_skip) if len(returns_skip) > 2 else 0.0
kurt_skip = kurtosis(returns_skip, fisher=False) if len(returns_skip) > 2 else 3.0
dsr_skip = deflated_sharpe_ratio(sr_skip, sr_benchmark, len(returns_skip), skew_skip, kurt_skip)

print(f'N_TRIALS: {N_TRIALS} (SKIP vs SMALL-PROFIT, keputusan biner)')
print(f'SR benchmark: {sr_benchmark:.4f}')
print(f'SKIP Sharpe: {sr_skip:.4f}')
print(f'SKIP Deflated Sharpe Ratio: {dsr_skip:.4f}')
if dsr_skip >= 0.5:
    print('>>> SIGNIFIKAN (DSR>=0.5) -- SKIP mengungguli SMALL-PROFIT bukan krn kebetulan.')
else:
    print('>>> TIDAK signifikan (DSR<0.5).')

Sharpe Ratio (TEST period, annualized): SMALL-PROFIT=3.6779, SKIP=4.0264
N_TRIALS: 2 (SKIP vs SMALL-PROFIT, keputusan biner)
SR benchmark: 0.1989
SKIP Sharpe: 4.0264
SKIP Deflated Sharpe Ratio: 1.0000
>>> SIGNIFIKAN (DSR>=0.5) -- SKIP mengungguli SMALL-PROFIT bukan krn kebetulan.


## 12. Perbandingan head-to-head FULL PERIOD (2019-2026): v13 murni vs v28 (live sekarang) vs v28+SKIP-exhaustion (usulan)

Supaya jelas kontribusi tiap lapis filter, bukan cuma TEST period.

In [25]:
# v13 murni: v28 engine dgn sr_source='none' (S/R filter dimatikan), exhaustion_mode='small_profit' (spt v13 asli)
v13_full = run_backtest_v31e(df, sr_source='none', exhaustion_mode='small_profit')
# v28 (LIVE SEKARANG): S/R filter aktif + exhaustion small_profit (spt usecase.py skrg persis)
v28_full = run_backtest_v31e(df, sr_source='both', exhaustion_mode='small_profit')
# v28+SKIP (USULAN): S/R filter aktif + exhaustion SKIP total
v28_skip_full = run_backtest_v31e(df, sr_source='both', exhaustion_mode='skip')

print('=== FULL PERIOD 2019-2026 (fixed lot 0.03, modal $2000, no kill-switch) ===')
print('v13 murni (tanpa S/R filter):        ', evaluate(v13_full, initial_equity=2000.0))
print('v28 (LIVE SEKARANG, S/R+small-profit):', evaluate(v28_full, initial_equity=2000.0))
print('v28+SKIP (USULAN, S/R+skip-exhaustion):', evaluate(v28_skip_full, initial_equity=2000.0))

=== FULL PERIOD 2019-2026 (fixed lot 0.03, modal $2000, no kill-switch) ===
v13 murni (tanpa S/R filter):         {'total_trades': 3466, 'win_rate_pct': 24.03, 'profit_factor': np.float64(0.81), 'net_pnl': np.float64(-3529.72), 'max_drawdown_pct': np.float64(-405.28)}
v28 (LIVE SEKARANG, S/R+small-profit): {'total_trades': 1801, 'win_rate_pct': 35.42, 'profit_factor': np.float64(1.19), 'net_pnl': np.float64(2195.66), 'max_drawdown_pct': np.float64(-121.72)}
v28+SKIP (USULAN, S/R+skip-exhaustion): {'total_trades': 1400, 'win_rate_pct': 36.64, 'profit_factor': np.float64(1.33), 'net_pnl': np.float64(2994.31), 'max_drawdown_pct': np.float64(-86.29)}
